# Abt-Buy published dataset benchmark

This notebook uses the **open Abt-Buy dataset** from the DeepMatcher SIGMOD'18 benchmark collection and runs `fuzzy_llm_matcher` on the test split.

Paper/dataset index:
- https://raw.githubusercontent.com/anhaidgroup/deepmatcher/master/Datasets.md

In [ ]:
from __future__ import annotations

import io
import time
import zipfile
from urllib.request import urlopen

import pandas as pd

from fuzzy_llm_matcher import evaluate_matches, match_tables

DATA_URL = (
    "http://pages.cs.wisc.edu/~anhai/data1/deepmatcher_data/"
    "Textual/Abt-Buy/abt_buy_exp_data.zip"
)

def load_abt_buy():
    with urlopen(DATA_URL) as resp:
        payload = resp.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        table_a = pd.read_csv(zf.open("exp_data/tableA.csv"))
        table_b = pd.read_csv(zf.open("exp_data/tableB.csv"))
        test_pairs = pd.read_csv(zf.open("exp_data/test.csv"))
    return table_a, table_b, test_pairs

table_a, table_b, test_pairs = load_abt_buy()
print(len(table_a), len(table_b), len(test_pairs))
table_a.head(3)

The dataset contains product names/descriptions from two sources (`tableA`, `tableB`) and labeled candidate pairs in `test.csv` with `label=1` for true matches.

In [ ]:
left_ids = set(test_pairs["ltable_id"].unique())
left = table_a[table_a["id"].isin(left_ids)][["id", "name"]].copy()
right = table_b[["id", "name"]].copy()

true_pairs = test_pairs[test_pairs["label"] == 1][["ltable_id", "rtable_id"]].rename(
    columns={"ltable_id": "left_id", "rtable_id": "right_id"}
)

start = time.perf_counter()
result = match_tables(
    left_df=left,
    right_df=right,
    left_on="name",
    right_on="name",
    left_id="id",
    right_id="id",
    top_k=5,
    use_llm=False,
)
elapsed = time.perf_counter() - start

ev = evaluate_matches(result, true_pairs, runtime_seconds=elapsed)
metrics = pd.Series(ev.to_dict())
metrics

These metrics show how the package performs on an external published benchmark split. You can tune thresholds (`high_threshold`, `medium_threshold`, `min_margin_high`) to trade precision vs recall.